# Решения: peeking и multireg

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

## Карта эталона

**Фокус:** Честный stop-rule и модель факторов.

Накопительные p-value демонстрируют риск peeking. Множественная регрессия дополняет A/B-анализ, но её коэффициенты описывают условные связи, а не автоматически причинные эффекты.

Эталон разделён на исполняемые секции в том же порядке, что `lesson.ipynb` и `homework.ipynb`. После каждой секции сверяйте не только значение, но и способ вычисления.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


## 0.1. Fixed-horizon stop-rule

До построения накопительных p-value задайте горизонт, уровень alpha и правило остановки. Промежуточная траектория служит демонстрацией риска, а не разрешением остановиться раньше.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
horizon_day = 30
alpha = 0.05
STOP_RULE = (
    'Собираем данные до конца дня 30 и выполняем одну итоговую проверку; '
    'промежуточные p-value не используются для досрочного решения.'
)
assert len(STOP_RULE) > 100

## Решение 1

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
def perm_p(frame, seed=0, n_iter=1200):
    rng = np.random.default_rng(seed)
    conv = frame['converted'].to_numpy()
    mask_b = frame['variant'].to_numpy() == 'B'
    obs = float(conv[mask_b].mean() - conv[~mask_b].mean())
    sims = np.empty(n_iter)
    for i in range(n_iter):
        perm = rng.permutation(conv)
        sims[i] = perm[mask_b].mean() - perm[~mask_b].mean()
    return float((np.abs(sims) >= abs(obs)).mean())

## Решение 2

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
rows = []

for d in range(1, 31):
    part = df[df['day'] <= d]
    if len(part) >= 200:
        rows.append({'day': d, 'p_value': perm_p(part, seed=500 + d)})

## Решение 3

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
daily = pd.DataFrame(rows)

sig_days = daily.loc[daily['p_value'] < 0.05, 'day']

first_sig_day = int(sig_days.iloc[0]) if len(sig_days) else -1

## Решение 4

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
PEEK_NOTE = (
    'Если смотреть p-value каждый день и останавливать эксперимент при первом p<0.05, '
    'растёт шанс ложноположительного вывода из-за многократной проверки.'
)

from sklearn.linear_model import LinearRegression

X = df[['variant_b', 'pages_viewed', 'prior_visits_30d', 'discount_pct', 'session_minutes', 'is_weekend']]

## Решение 5

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
y = df['converted']

model = LinearRegression().fit(X, y)

coef_table = pd.DataFrame({'feature': X.columns, 'coef': model.coef_}).sort_values('feature')

## Решение 6

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
r2 = float(model.score(X, y))

VARIANT_NOTE = (
    'Коэффициент при variant_b показывает среднее изменение предсказанной конверсии для B '
    'при фиксированных остальных признаках модели.'
)

LIN_NOTE = (
    'Линейная регрессия на target 0/1 проста для интерпретации, но может давать прогнозы вне [0, 1] '
    'и не учитывает нелинейность вероятности. Это учебная аппроксимация.'
)

## Решение 7

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
X_small = df[['variant_b', 'pages_viewed']]

r2_small = float(LinearRegression().fit(X_small, y).score(X_small, y))

r2_full = r2

## Решение 8

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
coef_variant = float(model.coef_[0])

coef_pages = float(model.coef_[1])

ANTI_PEEK = (
    'Фиксируем заранее длительность эксперимента и критерий остановки; '
    'не меняем гипотезу по ходу; одну итоговую проверку делаем после полного сбора данных.'
)

## Решение 9

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
COEF_LIMIT = (
    'Коэффициенты описывают связь внутри выбранной модели и признаков, но не доказывают причинность. '
    'Пропущенные факторы и коррелированные признаки могут смещать интерпретацию.'
)

print(first_sig_day, round(r2, 4))

print(coef_table)

## Проверка преподавателя

Запустите `Run All`. Эталон должен завершиться без исключений; итоговые числа должны совпадать при повторном запуске благодаря фиксированным seed. Текстовый вывод проверяется на согласованность с направлением uplift, p-value и границами CI.